# Matrice de richesse formelle — évaluer honnêtement ce qu'un solveur *décide réellement* (Rust)

Ce notebook présente la **méthodologie d'évaluation honnête** transverse à toute la série ArgumentAnalysis : comment construire une *matrice de richesse formelle* qui distingue ce qu'un moteur de raisonnement **décide pour de vrai** de ce qui n'est que décor.

> **SOTA, pas de contournement** : un composant décide réellement, ou il échoue bruyamment — jamais de fabrication.

## Le piège du théâtre (*fallback theater*)

Un système peut afficher **40 capacités formelles** — logique propositionnelle, FOL, modale, Dung, ASPIC+, ABA, défeasurable, bipolaire, ranking, probabiliste… — toutes **câblées** dans le code (un handler existe pour chacune), et **ne décider réellement que 5 choses**. Les 35 autres sont du théâtre : des handlers présents pour la demo, qui rendent un verdict creux ou affirmatif sans avoir fait tourner le moteur sous-jacent.

La question n'est pas *« quels solveurs sont câblés ? »* (un `grep` répond) mais *« quels solveurs ont produit une **décision réelle** sur ce corpus ? »*. Ce notebook construit l'instrument qui répond honnêtement.

**Navigation** : [<< 05b-Multi-Backend-Routing](Argumentation-05b-Multi-Backend-Routing-Python.ipynb) | [Index](README.md) | [Version Python](Argumentation-05c-Formal-Richness-Matrix-Python.ipynb) | [06-JTMS >>](Argumentation-06-JTMS-Python.ipynb)

## 1. Principe anti-théâtre : *wiring* ≠ *output*

L'anti-pattern fondateur (cf. `FP2_FALLBACK_THEATER_CENSUS.md`, `FP5_FORMAL_RICHNESS_MATRIX.md`) :

> **Mesurer la sortie, pas la présence.** Une capacité *câblée* (handler présent dans le code) n'est pas une capacité *exerçée* (verdict réel produit). Vendre un `grep` de présence comme une mesure de compétence = théâtre.

Concrètement, pour chaque couple (capacité × corpus), on ne lit **pas** le code source — on lit le **snapshot persisté** écrit par le moteur pendant l'exécution réelle : combien de mutations d'état a-t-il écrites ? a-t-il produit un verdict formel ? Ce compteur est une **preuve d'exécution**, pas une déclaration d'intention.

In [2]:
// Snapshot persisté d'une capacité sur un corpus, tel qu'un harness d'évaluation
// réelle l'écrirait sur disque. FP-23 classifie DEPUIS ces métriques persistées
// (offline, sans LLM, sans ré-exécuter le corpus) — pas depuis un grep de code.
#[allow(dead_code)]
struct CellMetrics {
    capability: String,
    corpus: String,
    wired: bool,                     // un handler est présent dans le code (grep dirait "oui")
    snapshot_count: usize,           // mutations d'état réelles (compteur persisté)
    verdict: Option<String>,         // verdict formel produit, ex. "accepted" / "rejected"
    fabricated_true: bool,           // SENTINELLE : verdict affirmatif fabriqué sans solveur réel
    subdimension_of: Option<String>, // se replie sur un axe parent déjà mesuré
}

// Constructeur canonique du carnet : wired vaut true (seule valeur utilisée aux
// sites d'appel) ; les défauts Python vivent ici, pas dans un derive(Default).
#[allow(dead_code)]
fn cm(capability: &str, corpus: &str, verdict: Option<&str>, snapshot_count: usize,
      fabricated_true: bool, subdimension_of: Option<&str>) -> CellMetrics {
    CellMetrics {
        capability: capability.to_string(),
        corpus: corpus.to_string(),
        wired: true,
        snapshot_count: snapshot_count,
        verdict: verdict.map(|v| v.to_string()),
        fabricated_true: fabricated_true,
        subdimension_of: subdimension_of.map(|s| s.to_string()),
    }
}

// La mesure NAÏVE (théâtre) : compte les handlers câblés.
// Vend la présence comme une compétence. Ne regarde AUCUNE sortie.
#[allow(dead_code)]
fn naive_wired_only(cells: &[CellMetrics]) -> (usize, usize) {
    (cells.iter().filter(|c| c.wired).count(), cells.len())
}


La mesure naïve ci-dessous est précisément ce qu'il faut **ne pas** faire. Elle répondrait « 24/24 capacités » alors que 10 cases sur 24 ne décident rien de substantif (14 substantive / 6 honest_absent / 3 unavailable / 1 théâtre — tally cellule 9). Construisons l'instrument honnête : une fonction de **classification par la sortie**.

## 2. Les quatre classes de verdict

FP-23 classe chaque cellule (capacité × corpus) en **quatre catégories** selon ce que le moteur a *réellement* produit — jamais selon ce qui est câblé :

| Classe | Condition (lue sur la sortie) | Verdict honnête |
|---|---|---|
| **`substantive`** | mutation d'état réelle (`count ≥ 1`) **ou** verdict formel produit | la capacité a **décidé** |
| **`honest_absent`** | câblée mais `count = 0`, aucun verdict, *non* fabriqué | le **corpus manque la structure** (ex. pas de programme défeasurable) → `fail-loud` vide, *pas un bug* |
| **`unavailable`** | se replie sur un axe parent déjà mesuré | sous-dimension d'un axe mesuré, *pas un laggard* |
| **`theatre_suspect`** | verdict affirmatif **fabriqué** (`fabricated_true`) sans solveur réel | **théâtre** → à réparer (rendre `fail-loud` ou câbler un vrai solveur) |

La distinction cruciale est entre `honest_absent` et `theatre_suspect` : un échec **honnête** (le corpus ne contient pas la structure attendue, le moteur le dit) est acceptable ; un verdict **fabriqué** ne l'est jamais.

In [3]:
// Snapshot persisté d'une capacité sur un corpus, tel qu'un harness d'évaluation
// réelle l'écrirait sur disque. FP-23 classifie DEPUIS ces métriques persistées
// (offline, sans LLM, sans ré-exécuter le corpus) — pas depuis un grep de code.
#[allow(dead_code)]
struct CellMetrics {
    capability: String,
    corpus: String,
    wired: bool,                     // un handler est présent dans le code (grep dirait "oui")
    snapshot_count: usize,           // mutations d'état réelles (compteur persisté)
    verdict: Option<String>,         // verdict formel produit, ex. "accepted" / "rejected"
    fabricated_true: bool,           // SENTINELLE : verdict affirmatif fabriqué sans solveur réel
    subdimension_of: Option<String>, // se replie sur un axe parent déjà mesuré
}

// Constructeur canonique du carnet : wired vaut true (seule valeur utilisée aux
// sites d'appel) ; les défauts Python vivent ici, pas dans un derive(Default).
#[allow(dead_code)]
fn cm(capability: &str, corpus: &str, verdict: Option<&str>, snapshot_count: usize,
      fabricated_true: bool, subdimension_of: Option<&str>) -> CellMetrics {
    CellMetrics {
        capability: capability.to_string(),
        corpus: corpus.to_string(),
        wired: true,
        snapshot_count: snapshot_count,
        verdict: verdict.map(|v| v.to_string()),
        fabricated_true: fabricated_true,
        subdimension_of: subdimension_of.map(|s| s.to_string()),
    }
}

const SUBSTANTIVE: &str = "substantive";
const HONEST_ABSENT: &str = "honest_absent";
const UNAVAILABLE: &str = "unavailable";
const THEATRE: &str = "theatre_suspect";

fn classify(cell: &CellMetrics) -> (String, String) {
    // Wiring != output : classifie DEPUIS la sortie (compteur/verdict/sentinelle),
    // jamais depuis la présence du handler. Fidèle à scripts/fp23_measure_unmeasured.py.
    if cell.subdimension_of.is_some() {
        let parent = cell.subdimension_of.as_deref().unwrap();
        return (UNAVAILABLE.to_string(), format!("sous-dimension de '{}'", parent));
    }
    if cell.fabricated_true {
        return (THEATRE.to_string(), "verdict affirmatif fabriqué sans solveur réel".to_string());
    }
    if cell.snapshot_count >= 1 || cell.verdict.is_some() {
        let detail = if cell.verdict.is_some() {
            "verdict réel".to_string()
        } else {
            format!("count={}", cell.snapshot_count)
        };
        return (SUBSTANTIVE.to_string(), detail);
    }
    // câblée, 0 mutation, aucun verdict, non fabriquée -> le corpus manque la structure
    (HONEST_ABSENT.to_string(), "corpus sans la structure requise (fail-loud empty)".to_string())
}

// Test rapide sur les quatre archétypes
let archetypes = [
    cm("pl", "doc_A", Some("decided"), 0, false, None),          // substantive
    cm("defeasible", "doc_A", None, 0, false, None),             // honest_absent
    cm("atms", "doc_A", None, 0, false, Some("jtms_belief")),    // unavailable
    cm("dung", "doc_A", Some("accepted"), 0, true, None),        // theatre
];
// evcxr : une boucle finale emet un execute_result () parasite -> enveloppee.
let _ = if true {
    for c in &archetypes {
        let (cls, detail) = classify(c);
        println!("{:<12} -> {:<14} ({})", c.capability, cls, detail);
    }
};


pl           -> substantive    (verdict réel)


defeasible   -> honest_absent  (corpus sans la structure requise (fail-loud empty))


atms         -> unavailable    (sous-dimension de 'jtms_belief')


dung         -> theatre_suspect (verdict affirmatif fabriqué sans solveur réel)


## 3. La sentinelle `fabricated_true`

Des quatre classes, la plus dangereuse est `theatre_suspect`. Le cas canonique (documenté FP-22 #1249) : un *fallback* `_python_dung_fallback` qui, **sans** la JVM Tweety, énumérait combinatoirement des extensions et renvoyait `"accepted"` — un verdict affirmatif **fabriqué**, indissociable d'un vrai verdict pour l'utilisateur.

Un échec honnête (`honest_absent`) est transparent : le système dit *« je n'ai pas trouvé de structure défeasurable »*. Un verdict fabriqué **ment** : il dit *« accepté »* sans avoir décidé. C'est pourquoi l'invariant anti-théâtre ne porte que sur ce drapeau — c'est le contrat de non-fabrication de toute la série.

In [4]:
// Snapshot persisté d'une capacité sur un corpus, tel qu'un harness d'évaluation
// réelle l'écrirait sur disque. FP-23 classifie DEPUIS ces métriques persistées
// (offline, sans LLM, sans ré-exécuter le corpus) — pas depuis un grep de code.
#[allow(dead_code)]
struct CellMetrics {
    capability: String,
    corpus: String,
    wired: bool,                     // un handler est présent dans le code (grep dirait "oui")
    snapshot_count: usize,           // mutations d'état réelles (compteur persisté)
    verdict: Option<String>,         // verdict formel produit, ex. "accepted" / "rejected"
    fabricated_true: bool,           // SENTINELLE : verdict affirmatif fabriqué sans solveur réel
    subdimension_of: Option<String>, // se replie sur un axe parent déjà mesuré
}

// Invariant anti-théâtre : AUCUN verdict affirmatif fabriqué.
// Wiring != output : on ne teste pas la présence, on teste la sentinelle de fabrication.
fn antitheatre_invariant<'a>(cells: &'a [CellMetrics]) -> Vec<&'a CellMetrics> {
    cells.iter().filter(|c| c.fabricated_true).collect()
}

#[allow(dead_code)]
fn assert_antitheatre(cells: &[CellMetrics]) -> bool {
    let culprits = antitheatre_invariant(cells);
    if !culprits.is_empty() {
        println!("INVARIANT VIOLÉ — {} théâtre(s) détecté(s) :", culprits.len());
        for c in culprits {
            println!("  - {} @ {} : verdict '{}' FABRIQUÉ (fabricated_true=True)",
                     c.capability, c.corpus, c.verdict.as_deref().unwrap_or("None"));
        }
        return false;
    }
    println!("INVARIANT OK — aucun verdict fabriqué (wiring == output côté sentinelles).");
    true
}


## 4. La matrice *capability × corpus*

On construit maintenant une matrice réaliste : **8 capacités × 3 corpus** (identifiants opaques `doc_A/B/C`, comme dans le rapport source — agrégat seul, aucun contenu de corpus). Les snapshots simulés couvrent chacune des quatre classes, y compris **un théâtre** (`dung_semantics` sur `doc_B`) que la sentinelle devra attraper.

In [5]:
// Snapshot persisté d'une capacité sur un corpus, tel qu'un harness d'évaluation
// réelle l'écrirait sur disque. FP-23 classifie DEPUIS ces métriques persistées
// (offline, sans LLM, sans ré-exécuter le corpus) — pas depuis un grep de code.
#[allow(dead_code)]
struct CellMetrics {
    capability: String,
    corpus: String,
    wired: bool,                     // un handler est présent dans le code (grep dirait "oui")
    snapshot_count: usize,           // mutations d'état réelles (compteur persisté)
    verdict: Option<String>,         // verdict formel produit, ex. "accepted" / "rejected"
    fabricated_true: bool,           // SENTINELLE : verdict affirmatif fabriqué sans solveur réel
    subdimension_of: Option<String>, // se replie sur un axe parent déjà mesuré
}

// Constructeur canonique du carnet : wired vaut true (seule valeur utilisée aux
// sites d'appel) ; les défauts Python vivent ici, pas dans un derive(Default).
#[allow(dead_code)]
fn cm(capability: &str, corpus: &str, verdict: Option<&str>, snapshot_count: usize,
      fabricated_true: bool, subdimension_of: Option<&str>) -> CellMetrics {
    CellMetrics {
        capability: capability.to_string(),
        corpus: corpus.to_string(),
        wired: true,
        snapshot_count: snapshot_count,
        verdict: verdict.map(|v| v.to_string()),
        fabricated_true: fabricated_true,
        subdimension_of: subdimension_of.map(|s| s.to_string()),
    }
}

const SUBSTANTIVE: &str = "substantive";
const HONEST_ABSENT: &str = "honest_absent";
const UNAVAILABLE: &str = "unavailable";
const THEATRE: &str = "theatre_suspect";

// La mesure NAÏVE (théâtre) : compte les handlers câblés.
// Vend la présence comme une compétence. Ne regarde AUCUNE sortie.
#[allow(dead_code)]
fn naive_wired_only(cells: &[CellMetrics]) -> (usize, usize) {
    (cells.iter().filter(|c| c.wired).count(), cells.len())
}

fn classify(cell: &CellMetrics) -> (String, String) {
    // Wiring != output : classifie DEPUIS la sortie (compteur/verdict/sentinelle),
    // jamais depuis la présence du handler. Fidèle à scripts/fp23_measure_unmeasured.py.
    if cell.subdimension_of.is_some() {
        let parent = cell.subdimension_of.as_deref().unwrap();
        return (UNAVAILABLE.to_string(), format!("sous-dimension de '{}'", parent));
    }
    if cell.fabricated_true {
        return (THEATRE.to_string(), "verdict affirmatif fabriqué sans solveur réel".to_string());
    }
    if cell.snapshot_count >= 1 || cell.verdict.is_some() {
        let detail = if cell.verdict.is_some() {
            "verdict réel".to_string()
        } else {
            format!("count={}", cell.snapshot_count)
        };
        return (SUBSTANTIVE.to_string(), detail);
    }
    // câblée, 0 mutation, aucun verdict, non fabriquée -> le corpus manque la structure
    (HONEST_ABSENT.to_string(), "corpus sans la structure requise (fail-loud empty)".to_string())
}

const CORPORA: [&str; 3] = ["doc_A", "doc_B", "doc_C"];

fn build_matrix() -> Vec<CellMetrics> {
    let mut cells = Vec::new();
    // (a) trois capacités qui décident vraiment sur tous les corpus : verdict réel
    for cap in ["pl_reasoning", "fol_reasoning", "aspic_plus_reasoning"] {
        for c in CORPORA {
            cells.push(cm(cap, c, Some("decided"), 0, false, None));
        }
    }
    // (b) ranking_semantics : substantive par compteur (mutation d'état réelle)
    for c in CORPORA {
        cells.push(cm("ranking_semantics", c, None, 3, false, None));
    }
    // (c) defeasible_logic : honest_absent (le corpus n'a pas de programme défeasurable)
    for c in CORPORA {
        cells.push(cm("defeasible_logic", c, None, 0, false, None));
    }
    // (d) modal_logic : honest_absent (corpus sans opérateurs modaux [])
    for c in CORPORA {
        cells.push(cm("modal_logic", c, None, 0, false, None));
    }
    // (e) atms_reasoning : unavailable (se replie sur jtms_belief déjà mesuré)
    for c in CORPORA {
        cells.push(cm("atms_reasoning", c, None, 0, false, Some("jtms_belief")));
    }
    // (f) dung_semantics : substantive partout SAUF doc_B = THEATRE
    //     (fallback python fabrique "accepted" sans la JVM Tweety — cas FP-22)
    for c in CORPORA {
        if c == "doc_B" {
            cells.push(cm("dung_semantics", c, Some("accepted"), 0, true, None));
        } else {
            cells.push(cm("dung_semantics", c, Some("accepted"), 0, false, None));
        }
    }
    cells
}

let matrix = build_matrix();

// La mesure NAÏVE (théâtre) vs la mesure honnête
let (naive_wired, total) = naive_wired_only(&matrix);
println!("Mesure naïve (grep de présence) : {}/{} capacités 'couvertes'", naive_wired, total);
println!("  -> cette mesure est MENSONGÈRE : elle compte dung@doc_B comme couvert.");
println!();

// Tally honnête par classe (liste explicite, jamais une HashMap itérée)
let order = [SUBSTANTIVE, HONEST_ABSENT, UNAVAILABLE, THEATRE];
println!("Tally honnête (capability × corpus, {} cellules) :", total);
let _ = if true {
    for k in order {
        let n = matrix.iter().filter(|c| classify(c).0 == k).count();
        println!("  {:<14} : {}", k, n);
    }
};


Mesure naïve (grep de présence) : 24/24 capacités 'couvertes'


  -> cette mesure est MENSONGÈRE : elle compte dung@doc_B comme couvert.


Tally honnête (capability × corpus, 24 cellules) :


  substantive    : 14


  honest_absent  : 6


  unavailable    : 3


  theatre_suspect : 1


### Lecture de la matrice

La mesure naïve vendrait **24/24 cases couvertes** (tout est câblé). La mesure honnête révèle que **1 case est du théâtre** (`dung_semantics @ doc_B`) : le système *prétend* décider alors qu'il fabrique. C'est exactement l'écart que FP-5 a été construit pour mesurer.

## 5. Diagnostic des laggards — honnête vs théâtre

Pour chaque cellule non-`substantive`, on produit un **diagnostic root-cause**. La discipline :

- `honest_absent` → **OK** : le corpus manque la structure, le moteur le dit (`fail-loud` vide). Pas un bug.
- `unavailable` → **OK** : sous-dimension d'un axe parent déjà mesuré. Pas un laggard.
- `theatre_suspect` → **À RÉPARER** : verdict fabriqué. Deux voies honnêtes : rendre `fail-loud` (comme FP-22 l'a fait pour `_python_dung_fallback`) **ou** câbler un vrai solveur.

In [6]:
// Snapshot persisté d'une capacité sur un corpus, tel qu'un harness d'évaluation
// réelle l'écrirait sur disque. FP-23 classifie DEPUIS ces métriques persistées
// (offline, sans LLM, sans ré-exécuter le corpus) — pas depuis un grep de code.
#[allow(dead_code)]
struct CellMetrics {
    capability: String,
    corpus: String,
    wired: bool,                     // un handler est présent dans le code (grep dirait "oui")
    snapshot_count: usize,           // mutations d'état réelles (compteur persisté)
    verdict: Option<String>,         // verdict formel produit, ex. "accepted" / "rejected"
    fabricated_true: bool,           // SENTINELLE : verdict affirmatif fabriqué sans solveur réel
    subdimension_of: Option<String>, // se replie sur un axe parent déjà mesuré
}

// Constructeur canonique du carnet : wired vaut true (seule valeur utilisée aux
// sites d'appel) ; les défauts Python vivent ici, pas dans un derive(Default).
#[allow(dead_code)]
fn cm(capability: &str, corpus: &str, verdict: Option<&str>, snapshot_count: usize,
      fabricated_true: bool, subdimension_of: Option<&str>) -> CellMetrics {
    CellMetrics {
        capability: capability.to_string(),
        corpus: corpus.to_string(),
        wired: true,
        snapshot_count: snapshot_count,
        verdict: verdict.map(|v| v.to_string()),
        fabricated_true: fabricated_true,
        subdimension_of: subdimension_of.map(|s| s.to_string()),
    }
}

const SUBSTANTIVE: &str = "substantive";
const HONEST_ABSENT: &str = "honest_absent";
const UNAVAILABLE: &str = "unavailable";
const THEATRE: &str = "theatre_suspect";

fn classify(cell: &CellMetrics) -> (String, String) {
    // Wiring != output : classifie DEPUIS la sortie (compteur/verdict/sentinelle),
    // jamais depuis la présence du handler. Fidèle à scripts/fp23_measure_unmeasured.py.
    if cell.subdimension_of.is_some() {
        let parent = cell.subdimension_of.as_deref().unwrap();
        return (UNAVAILABLE.to_string(), format!("sous-dimension de '{}'", parent));
    }
    if cell.fabricated_true {
        return (THEATRE.to_string(), "verdict affirmatif fabriqué sans solveur réel".to_string());
    }
    if cell.snapshot_count >= 1 || cell.verdict.is_some() {
        let detail = if cell.verdict.is_some() {
            "verdict réel".to_string()
        } else {
            format!("count={}", cell.snapshot_count)
        };
        return (SUBSTANTIVE.to_string(), detail);
    }
    // câblée, 0 mutation, aucun verdict, non fabriquée -> le corpus manque la structure
    (HONEST_ABSENT.to_string(), "corpus sans la structure requise (fail-loud empty)".to_string())
}

const CORPORA: [&str; 3] = ["doc_A", "doc_B", "doc_C"];

fn build_matrix() -> Vec<CellMetrics> {
    let mut cells = Vec::new();
    // (a) trois capacités qui décident vraiment sur tous les corpus : verdict réel
    for cap in ["pl_reasoning", "fol_reasoning", "aspic_plus_reasoning"] {
        for c in CORPORA {
            cells.push(cm(cap, c, Some("decided"), 0, false, None));
        }
    }
    // (b) ranking_semantics : substantive par compteur (mutation d'état réelle)
    for c in CORPORA {
        cells.push(cm("ranking_semantics", c, None, 3, false, None));
    }
    // (c) defeasible_logic : honest_absent (le corpus n'a pas de programme défeasurable)
    for c in CORPORA {
        cells.push(cm("defeasible_logic", c, None, 0, false, None));
    }
    // (d) modal_logic : honest_absent (corpus sans opérateurs modaux [])
    for c in CORPORA {
        cells.push(cm("modal_logic", c, None, 0, false, None));
    }
    // (e) atms_reasoning : unavailable (se replie sur jtms_belief déjà mesuré)
    for c in CORPORA {
        cells.push(cm("atms_reasoning", c, None, 0, false, Some("jtms_belief")));
    }
    // (f) dung_semantics : substantive partout SAUF doc_B = THEATRE
    //     (fallback python fabrique "accepted" sans la JVM Tweety — cas FP-22)
    for c in CORPORA {
        if c == "doc_B" {
            cells.push(cm("dung_semantics", c, Some("accepted"), 0, true, None));
        } else {
            cells.push(cm("dung_semantics", c, Some("accepted"), 0, false, None));
        }
    }
    cells
}

// Le dict ACTIONS n'est lu que par clé : un match par cas, pas d'ordre à préserver.
fn action_for(cls: &str) -> &'static str {
    match cls {
        HONEST_ABSENT => "OK (fail-loud empty) — le corpus manque la structure, pas un bug",
        UNAVAILABLE => "OK — se replie sur un axe parent mesuré, pas un laggard",
        _ => "À RÉPARER — verdict fabriqué : rendre fail-loud OU câbler un vrai solveur",
    }
}

fn diagnose(cells: &[CellMetrics]) {
    println!("Diagnostic des cellules non-substantives :");
    println!("{}", "-".repeat(78));
    for c in cells {
        let (cls, detail) = classify(c);
        if cls == SUBSTANTIVE {
            continue;
        }
        println!("  {:<22} @ {:<6} : {:<14} — {}", c.capability, c.corpus, cls, detail);
        println!("      -> {}", action_for(&cls));
    }
}

diagnose(&build_matrix());


Diagnostic des cellules non-substantives :


------------------------------------------------------------------------------


  defeasible_logic       @ doc_A  : honest_absent  — corpus sans la structure requise (fail-loud empty)


      -> OK (fail-loud empty) — le corpus manque la structure, pas un bug


  defeasible_logic       @ doc_B  : honest_absent  — corpus sans la structure requise (fail-loud empty)


      -> OK (fail-loud empty) — le corpus manque la structure, pas un bug


... (12 outputs left out)

  atms_reasoning         @ doc_C  : unavailable    — sous-dimension de 'jtms_belief'


      -> OK — se replie sur un axe parent mesuré, pas un laggard


  dung_semantics         @ doc_B  : theatre_suspect — verdict affirmatif fabriqué sans solveur réel


      -> À RÉPARER — verdict fabriqué : rendre fail-loud OU câbler un vrai solveur


## 6. Réparation et invariant end-to-end

On applique le diagnostic : le théâtre `dung_semantics @ doc_B` est réparé selon la voie **`fail-loud`** (la plus honnête quand on n'a pas de solveur réel sous la main) — on retire le verdict fabriqué et on déclare honnêtement l'absence de décision. On re-run l'invariant : il doit maintenant passer.

In [7]:
// Snapshot persisté d'une capacité sur un corpus, tel qu'un harness d'évaluation
// réelle l'écrirait sur disque. FP-23 classifie DEPUIS ces métriques persistées
// (offline, sans LLM, sans ré-exécuter le corpus) — pas depuis un grep de code.
#[allow(dead_code)]
struct CellMetrics {
    capability: String,
    corpus: String,
    wired: bool,                     // un handler est présent dans le code (grep dirait "oui")
    snapshot_count: usize,           // mutations d'état réelles (compteur persisté)
    verdict: Option<String>,         // verdict formel produit, ex. "accepted" / "rejected"
    fabricated_true: bool,           // SENTINELLE : verdict affirmatif fabriqué sans solveur réel
    subdimension_of: Option<String>, // se replie sur un axe parent déjà mesuré
}

// Constructeur canonique du carnet : wired vaut true (seule valeur utilisée aux
// sites d'appel) ; les défauts Python vivent ici, pas dans un derive(Default).
#[allow(dead_code)]
fn cm(capability: &str, corpus: &str, verdict: Option<&str>, snapshot_count: usize,
      fabricated_true: bool, subdimension_of: Option<&str>) -> CellMetrics {
    CellMetrics {
        capability: capability.to_string(),
        corpus: corpus.to_string(),
        wired: true,
        snapshot_count: snapshot_count,
        verdict: verdict.map(|v| v.to_string()),
        fabricated_true: fabricated_true,
        subdimension_of: subdimension_of.map(|s| s.to_string()),
    }
}

const SUBSTANTIVE: &str = "substantive";
const HONEST_ABSENT: &str = "honest_absent";
const UNAVAILABLE: &str = "unavailable";
const THEATRE: &str = "theatre_suspect";

// La mesure NAÏVE (théâtre) : compte les handlers câblés.
// Vend la présence comme une compétence. Ne regarde AUCUNE sortie.
#[allow(dead_code)]
fn naive_wired_only(cells: &[CellMetrics]) -> (usize, usize) {
    (cells.iter().filter(|c| c.wired).count(), cells.len())
}

fn classify(cell: &CellMetrics) -> (String, String) {
    // Wiring != output : classifie DEPUIS la sortie (compteur/verdict/sentinelle),
    // jamais depuis la présence du handler. Fidèle à scripts/fp23_measure_unmeasured.py.
    if cell.subdimension_of.is_some() {
        let parent = cell.subdimension_of.as_deref().unwrap();
        return (UNAVAILABLE.to_string(), format!("sous-dimension de '{}'", parent));
    }
    if cell.fabricated_true {
        return (THEATRE.to_string(), "verdict affirmatif fabriqué sans solveur réel".to_string());
    }
    if cell.snapshot_count >= 1 || cell.verdict.is_some() {
        let detail = if cell.verdict.is_some() {
            "verdict réel".to_string()
        } else {
            format!("count={}", cell.snapshot_count)
        };
        return (SUBSTANTIVE.to_string(), detail);
    }
    // câblée, 0 mutation, aucun verdict, non fabriquée -> le corpus manque la structure
    (HONEST_ABSENT.to_string(), "corpus sans la structure requise (fail-loud empty)".to_string())
}

// Invariant anti-théâtre : AUCUN verdict affirmatif fabriqué.
// Wiring != output : on ne teste pas la présence, on teste la sentinelle de fabrication.
fn antitheatre_invariant<'a>(cells: &'a [CellMetrics]) -> Vec<&'a CellMetrics> {
    cells.iter().filter(|c| c.fabricated_true).collect()
}

#[allow(dead_code)]
fn assert_antitheatre(cells: &[CellMetrics]) -> bool {
    let culprits = antitheatre_invariant(cells);
    if !culprits.is_empty() {
        println!("INVARIANT VIOLÉ — {} théâtre(s) détecté(s) :", culprits.len());
        for c in culprits {
            println!("  - {} @ {} : verdict '{}' FABRIQUÉ (fabricated_true=True)",
                     c.capability, c.corpus, c.verdict.as_deref().unwrap_or("None"));
        }
        return false;
    }
    println!("INVARIANT OK — aucun verdict fabriqué (wiring == output côté sentinelles).");
    true
}

const CORPORA: [&str; 3] = ["doc_A", "doc_B", "doc_C"];

fn build_matrix() -> Vec<CellMetrics> {
    let mut cells = Vec::new();
    // (a) trois capacités qui décident vraiment sur tous les corpus : verdict réel
    for cap in ["pl_reasoning", "fol_reasoning", "aspic_plus_reasoning"] {
        for c in CORPORA {
            cells.push(cm(cap, c, Some("decided"), 0, false, None));
        }
    }
    // (b) ranking_semantics : substantive par compteur (mutation d'état réelle)
    for c in CORPORA {
        cells.push(cm("ranking_semantics", c, None, 3, false, None));
    }
    // (c) defeasible_logic : honest_absent (le corpus n'a pas de programme défeasurable)
    for c in CORPORA {
        cells.push(cm("defeasible_logic", c, None, 0, false, None));
    }
    // (d) modal_logic : honest_absent (corpus sans opérateurs modaux [])
    for c in CORPORA {
        cells.push(cm("modal_logic", c, None, 0, false, None));
    }
    // (e) atms_reasoning : unavailable (se replie sur jtms_belief déjà mesuré)
    for c in CORPORA {
        cells.push(cm("atms_reasoning", c, None, 0, false, Some("jtms_belief")));
    }
    // (f) dung_semantics : substantive partout SAUF doc_B = THEATRE
    //     (fallback python fabrique "accepted" sans la JVM Tweety — cas FP-22)
    for c in CORPORA {
        if c == "doc_B" {
            cells.push(cm("dung_semantics", c, Some("accepted"), 0, true, None));
        } else {
            cells.push(cm("dung_semantics", c, Some("accepted"), 0, false, None));
        }
    }
    cells
}

// Avant réparation : l'invariant doit ÉCHOUER (1 théâtre)
let mut matrix = build_matrix();
println!("=== Avant réparation ===");
let _ok_before = assert_antitheatre(&matrix);
println!();

// Réparation fail-loud : on ne fabrique plus, on déclare l'absence honnêtement
for c in matrix.iter_mut() {
    if c.capability == "dung_semantics" && c.corpus == "doc_B" {
        c.fabricated_true = false;
        c.verdict = None;        // on retire le verdict fabriqué
        c.snapshot_count = 0;    // aucune mutation réelle -> deviendra honest_absent
    }
}

// Après réparation : l'invariant doit PASSER ; dung@doc_B reclassifie en honest_absent
println!("=== Après réparation (fail-loud) ===");
let _ok_after = assert_antitheatre(&matrix);
println!();
let order = [SUBSTANTIVE, HONEST_ABSENT, UNAVAILABLE, THEATRE];
let tally2: Vec<String> = order.iter()
    .map(|k| format!("{}={}", k, matrix.iter().filter(|c| classify(c).0 == *k).count()))
    .collect();
println!("Nouveau tally : {}", tally2.join(" · "));
let _ = {
    // Bloc : evcxr ne peut persister ni un emprunt non-'static (dung_b) ni une
    // expression finale () au niveau supérieur -- reference et valeur vivent
    // et meurent dans le bloc.
    let dung_b = matrix.iter().find(|c| c.capability == "dung_semantics" && c.corpus == "doc_B").unwrap();
    println!();
    println!("dung_semantics @ doc_B reclassifiée : {} (le théâtre est devenu un échec honnête).",
             classify(dung_b).0);
};


=== Avant réparation ===


INVARIANT VIOLÉ — 1 théâtre(s) détecté(s) :


  - dung_semantics @ doc_B : verdict 'accepted' FABRIQUÉ (fabricated_true=True)


=== Après réparation (fail-loud) ===


INVARIANT OK — aucun verdict fabriqué (wiring == output côté sentinelles).


Nouveau tally : substantive=14 · honest_absent=7 · unavailable=3 · theatre_suspect=0


dung_semantics @ doc_B reclassifiée : honest_absent (le théâtre est devenu un échec honnête).


## À retenir

La matrice de richesse formelle est l'**instrument de vérité** de toute la série ArgumentAnalysis :

- Elle **mesure** (sortie) au lieu de **déclarer** (présence) — *wiring ≠ output*.
- Elle **distingue l'échec honnête du théâtre** : un `fail-loud` vide vaut mieux qu'un verdict fabriqué.
- Elle rend le diagnostic **actionnable** : chaque cellule non-substantive a une cause racine et une réponse (OK / à réparer).
- Elle **croise candidates A et B** : le routeur multi-backend (candidate A, [Argumentation-05b-Multi-Backend-Routing-Python](Argumentation-05b-Multi-Backend-Routing-Python.ipynb)) décide ou échoue bruyamment ; la restitution 3-actes (candidate B, [Argumentation-08d-Restitution-3-Actes-Python](Argumentation-08d-Restitution-3-Actes-Python.ipynb)) ne narre que des verdicts réels — **ce notebook est le mètre qui garantit que les deux ne vendent que de la décision vraie**.

> **Références source** (`2025-Epita-Intelligence-Symbolique`) : `docs/reports/FP5_FORMAL_RICHNESS_MATRIX.md` (FP-5 #1196), `scripts/fp23_measure_unmeasured.py` (FP-23 #1250), `docs/reports/FP2_FALLBACK_THEATER_CENSUS.md` (FP-22 #1249), Epic #1191 *depth-parity*. Voir issue CoursIA #2137.

## Vérifiez votre compréhension

> Stubs à compléter (règle C.1 : jamais `raise`/`assert False`/`1/0` — le notebook doit s'exécuter de bout en bout). Réutilisez `CellMetrics`, `classify`, `assert_antitheatre`, `build_matrix`. La méthode de chaque exercice est déjà démontrée dans le carnet :

- **Exercice 1** (capacité honnête-absente) → la section 2 a classé les quatre verdicts (`defeasible -> honest_absent`, corpus sans la structure requise) et la section 6 a montré l'invariant : un échec honnête (`fail-loud` vide) ne violente pas la sentinelle.
- **Exercice 2** (injecter un théâtre et le faire attraper) → la section 6 l'a exécuté : le théâtre injecté est attrapé (`dung_semantics @ doc_B : verdict 'accepted' FABRIQUÉ (fabricated_true=True)`), puis la réparation fail-loud fait tomber le tally à `theatre_suspect=0`.
- **Exercice 3** (quatrième corpus) → la section 4 a construit la matrice `capability × corpus` et son tally honnête (`substantive 14 · honest_absent 6 · unavailable 3 · theatre_suspect 1`, contre un grep de présence qui dirait « 24/24 couvertes ») ; un corpus de plus est une colonne de plus et un tally à recompter.


### Exercice 1 — Ajouter une capacité honnête-absente

Ajoutez `bipolar_argumentation` aux trois corpus, câblée mais sans programme bipolaire dans le corpus (`snapshot_count=0`, pas de verdict, pas fabriquée). Vérifiez qu'elle classifie en `honest_absent` et que l'invariant anti-théâtre reste OK (un échec honnête ne violente pas la sentinelle).

In [8]:
fn exercice_1() {
    // Ajoute bipolar_argumentation (honest_absent x3) et re-valide l'invariant.
    // TODO etudiant : completer
    // 1. etendre la matrice avec 3 cm("bipolar_argumentation", c, None, 0, false, None)
    // 2. classifier la nouvelle capacite sur chaque corpus
    // 3. re-executer assert_antitheatre sur la matrice etendue
    println!("Exercice 1 a completer");
}
exercice_1();


Exercice 1 a completer


### Exercice 2 — Injecter un théâtre et le faire attraper

Injectez un théâtre : `weighted_argumentation @ doc_C`, câblée, avec `fabricated_true=True` et un `verdict="accepted"` fabriqué. L'invariant anti-théâtre doit le **détecter** (le nommer explicitement), puis vous le réparerez en `fail-loud`.

In [9]:
fn exercice_2() {
    // Injecte un theatre weighted_argumentation@doc_C, le fait attraper, puis repare.
    // TODO etudiant : completer
    // 1. ajouter cm("weighted_argumentation", "doc_C", Some("accepted"), 0, true, None)
    // 2. verifier que assert_antitheatre ECHOUE et nomme weighted_argumentation@doc_C
    // 3. le reparer en fail-loud (fabricated_true=false, verdict=None, snapshot_count=0)
    // 4. re-verifier que l'invariant passe
    println!("Exercice 2 a completer");
}
exercice_2();


Exercice 2 a completer


### Exercice 3 — Étendre à un quatrième corpus et mesurer le tally global

Ajoutez un corpus `doc_D` (par exemple : `pl_reasoning`, `fol_reasoning` substantive ; `defeasible_logic` honest_absent ; `dung_semantics` substantive). Calculez le tally global sur les 4 corpus et affichez le pourcentage de cellules `substantive`. Bonus : que faudrait-il pour qu'un corpus fasse passer `defeasible_logic` de `honest_absent` à `substantive` ?

In [10]:
fn exercice_3() {
    // Etend la matrice a doc_D et calcule le tally global + % substantive.
    // TODO etudiant : completer
    // 1. ajouter les CellMetrics pour doc_D selon l'enonce
    // 2. recompter le tally global (substantive / honest_absent / unavailable / theatre)
    // 3. afficher le pourcentage de cellules substantive
    // Bonus : repondre en commentaire -- quel contenu de corpus active defeasible_logic ?
    println!("Exercice 3 a completer");
}
exercice_3();


Exercice 3 a completer


## Références et provenance

**Sources et provenance** : Candidate C de la distillation Argumentum (#2137). Source distillée : `docs/reports/FP5_FORMAL_RICHNESS_MATRIX.md` (FP-5 #1196, Epic #1191 *depth-parity*) et `scripts/fp23_measure_unmeasured.py` (FP-23 #1250), du dépôt `2025-Epita-Intelligence-Symbolique`.


## Pour aller plus loin

- [Index de la sous-série ArgumentAnalysis](README.md) — la place du mètre dans le parcours.
- [Index de la série SymbolicAI](../README.md) — le fil complet.

Le carnet [05b (routage multi-backend)](Argumentation-05b-Multi-Backend-Routing-Python.ipynb) est la candidate A que cette matrice audite ; le carnet [08d (restitution 3 actes)](Argumentation-08d-Restitution-3-Actes-Python.ipynb) est la candidate B ; le carnet [05 (vérification formelle)](Argumentation-05-Formal-Verification-Python.ipynb) montre ce que « décider substantivement » veut dire, logique par logique.
